<a href="https://colab.research.google.com/github/ASHVISH03/GEN-AI-LAB-EXPERIMENTS/blob/main/GenAiExpt7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 959.1 kB/s eta 0:00:00


In [2]:
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# 1. Create a small domain-specific dataset

texts = [
    "The football team won the match",
    "The player scored a goal in the final",
    "The cricket match was exciting",
    "The team qualified for the tournament",
    "The coach announced the final squad",
    "The basketball player scored 30 points",
    "The tennis player won the championship",
    "The football league starts next month",

    "Python is widely used in machine learning",
    "Artificial intelligence is transforming industries",
    "Deep learning uses neural networks",
    "Transformers are used in NLP",
    "Machine learning models require training data",
    "Natural language processing deals with text",
    "Computer vision analyzes images",
    "Large language models can generate text"
]

labels = [
    0, 0, 0, 0, 0, 0, 0, 0,
    1, 1, 1, 1, 1, 1, 1, 1
]

dataset = Dataset.from_dict({
    "text": texts,
    "label": labels
})

print(dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 16
})


In [4]:
# 2. Load the pretrained tokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [5]:
# 3. Tokenize the dataset

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenized successfully.")
print(tokenized_dataset)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Dataset tokenized successfully.
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16
})


In [6]:
# 4. Load the pretrained BERT model

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("BERT classification model loaded successfully.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT classification model loaded successfully.


In [8]:
# 5. Define training arguments

training_args = TrainingArguments(
    output_dir="./fine_tuned_bert",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

print("Training arguments configured successfully.")

Training arguments configured successfully.


In [9]:
# 6. Create the Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("Trainer created successfully.")

Trainer created successfully.


In [10]:
# 7. Fine-tune the model

print("Starting fine-tuning...")
trainer.train()

print("Fine-tuning completed successfully.")

Starting fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.675558
2,0.704120
3,0.740098
4,0.628736
5,0.538423
6,0.471336
7,0.540672
8,0.639525
9,0.422971
10,0.609763


Fine-tuning completed successfully.


In [11]:
# 8. Save the fine-tuned model

save_path = "./fine_tuned_bert"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Fine-tuned model saved successfully.")
print("Model path:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully.
Model path: ./fine_tuned_bert


In [12]:
# 9. Test the fine-tuned model

classifier = pipeline(
    "text-classification",
    model=save_path,
    tokenizer=save_path
)

label_names = {
    "LABEL_0": "Sports",
    "LABEL_1": "Technology"
}

test_texts = [
    "The team won the football championship",
    "Artificial intelligence is growing rapidly"
]

print("PREDICTIONS")
print("=" * 40)

for text in test_texts:
    result = classifier(text)[0]

    label = label_names.get(
        result["label"],
        result["label"]
    )

    print("\nText:", text)
    print("Predicted Category:", label)
    print("Confidence:", f"{result['score']:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PREDICTIONS

Text: The team won the football championship
Predicted Category: Sports
Confidence: 0.7285

Text: Artificial intelligence is growing rapidly
Predicted Category: Technology
Confidence: 0.5724
